# Neural Network classification with pyTorch

## 1. Making Data

In [2]:
import sklearn
from sklearn.datasets import make_circles
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
import ipywidgets

In [3]:
# Make 1000 samples
n_samples = 1000

X, y = make_circles(n_samples,
                    noise=0.03,
                    random_state=42)
# Create circle
def cir_plot(noise):
  X, y = make_circles(n_samples,
                      noise=noise,
                      random_state=42)
  plt.style.use('dark_background')
  plt.scatter(x=X[:, 0],
              y=X[:, 1],
              c=y,
              cmap=plt.cm.RdYlBu)

In [4]:
ipywidgets.interact(cir_plot, noise=(0, 1, 0.01))

interactive(children=(FloatSlider(value=0.0, description='noise', max=1.0, step=0.01), Output()), _dom_classes…

<function __main__.cir_plot(noise)>

In [5]:
len(X), len(y)

(1000, 1000)

In [6]:
X.shape, y.shape

((1000, 2), (1000,))

In [7]:
X[:3], y[:3]

(array([[ 0.75424625,  0.23148074],
        [-0.75615888,  0.15325888],
        [-0.81539193,  0.17328203]]),
 array([1, 1, 1]))

In [8]:
circles = pd.DataFrame({"X1": X[:, 0],
                        "X2": X[:, 1],
                          "Label": y})
circles.head()

,X1,X2,Label
0,0.754246,0.231481,1
1,-0.756159,0.153259,1
2,-0.815392,0.173282,1
3,-0.393731,0.692883,1
4,0.442208,-0.896723,0


### 1.1 Check input and output shapes

In [9]:
X[:, :4]

array([[ 0.75424625,  0.23148074],
       [-0.75615888,  0.15325888],
       [-0.81539193,  0.17328203],
       ...,
       [-0.13690036, -0.81001183],
       [ 0.67036156, -0.76750154],
       [ 0.28105665,  0.96382443]])

In [10]:
# View the first example of features and labels
X_sample = X[0]
y_sample = y[0]
print(f'Value for single sample of X: {X_sample}, and the same for y: {y_sample}')
print(f'Shape for single sample of X: {X_sample.shape}, and the same for y: {y_sample.shape}')

Value for single sample of X: [0.75424625 0.23148074], and the same for y: 1
Shape for single sample of X: (2,), and the same for y: ()


### 1.2 Turn data into tensor and create train and test splits

In [11]:
type(X)

numpy.ndarray

In [12]:
# Turn data into tensors
X = torch.from_numpy(X).type(torch.float)
y = torch.from_numpy(y).type(torch.float)
X[:3], y[:3]

(tensor([[ 0.7542,  0.2315],
         [-0.7562,  0.1533],
         [-0.8154,  0.1733]]),
 tensor([1., 1., 1.]))

In [13]:
type(X), X.dtype, y.dtype

(torch.Tensor, torch.float32, torch.float32)

In [14]:
# Split data into train and test set
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X,
                                                    y,
                                                    test_size=0.2,
                                                    random_state=42)


In [15]:
len(X_train), len(X_test)

(800, 200)

## 2. Building a model

In [16]:
# Make device agnostic code
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [17]:
class CirclePredV0(nn.Module):
  def __init__(self):
    super().__init__()

    # Create nn layer capable of handling the shape of our data
    self.layer_1 = nn.Linear(in_features=2, out_features=5)
    self.layer_2 = nn.Linear(in_features=5, out_features=1)

  # Define a forward layer
  def forward(self, x):
    return self.layer_2(self.layer_1(x))

model_0 = CirclePredV0().to(device)
model_0

CirclePredV0(
  (layer_1): Linear(in_features=2, out_features=5, bias=True)
  (layer_2): Linear(in_features=5, out_features=1, bias=True)
)

In [18]:
next(model_0.parameters()).device

device(type='cuda', index=0)

In [19]:
model_0 = nn.Sequential(
    nn.Linear(in_features=2, out_features=5),
    nn.Linear(in_features=5, out_features=1)
).to(device)


model_0

Sequential(
  (0): Linear(in_features=2, out_features=5, bias=True)
  (1): Linear(in_features=5, out_features=1, bias=True)
)

In [20]:
model_0.state_dict()

OrderedDict([('0.weight',
              tensor([[ 0.6725, -0.5808],
                      [-0.5035,  0.2437],
                      [ 0.0454, -0.4551],
                      [ 0.6536,  0.5939],
                      [-0.3643,  0.5693]], device='cuda:0')),
             ('0.bias',
              tensor([ 0.0935,  0.2204, -0.3374,  0.6583,  0.5710], device='cuda:0')),
             ('1.weight',
              tensor([[-0.0298,  0.4125, -0.3755,  0.0222,  0.1815]], device='cuda:0')),
             ('1.bias', tensor([-0.1042], device='cuda:0'))])

In [21]:
# Make predictions
with torch.inference_mode():
  untrained_preds = model_0(X_test.to(device))
print(f"length: {len(untrained_preds)}, shape: {untrained_preds.shape}")
print(f"preds is: \n{torch.round(untrained_preds[:10])} \nand ground true is: \n{y_test[:10]}")


length: 200, shape: torch.Size([200, 1])
preds is: 
tensor([[1.],
        [1.],
        [0.],
        [1.],
        [-0.],
        [-0.],
        [0.],
        [0.],
        [0.],
        [1.]], device='cuda:0') 
and ground true is: 
tensor([1., 0., 1., 0., 1., 1., 0., 0., 1., 0.])


## 2.1 Setup loss function and optimizer

In [22]:
# Setup the loss function
loss_fn = nn.BCEWithLogitsLoss()

optimizer = torch.optim.SGD(params=model_0.parameters(),
                            lr=0.01)

In [ ]:
# Calculate accuracy
def accuracy_fn(y_true, y_pred):
  correct = torch.eq(y_ture, y_pred).sum().item()
  acc = (correct/len(y_true)) * 100
  return acc
